# RankFlow Demo

**rankflow** visualizes and evaluates how document ranks change across retrieval and re-ranking steps -- a common pattern in RAG (Retrieval Augmented Generation) pipelines.

A typical pipeline retrieves candidate documents with a fast method (e.g. BM25), then re-ranks them with one or more neural models (semantic search, cross-encoder). RankFlow shows exactly how each document moves through these stages and quantifies retrieval quality at every step.

Repository: [github.com/izikeros/rankflow](https://github.com/izikeros/rankflow)

## Setup

In [ ]:
import numpy as np
import pandas as pd

from rankflow import RankFlow, BatchRankFlow, PlotConfig

## 1. Quick Start -- Basic Rank Evolution Plot

The simplest use case: pass a 2-D array of shape `(n_steps, n_documents)` where each value is the rank of that document at that step. Lower rank = better position.

In [ ]:
ranks = np.array([
    [0, 1, 2, 3],  # BM25 retrieval
    [2, 0, 3, 1],  # after semantic re-ranking
    [1, 2, 0, 3],  # after cross-encoder re-ranking
])

rf = RankFlow(
    ranks=ranks,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
)
rf.plot()

## 2. Creating RankFlow from a pandas DataFrame

If your rank data is already in a DataFrame (rows = steps, columns = documents), you can pass it directly. The index becomes step labels and the columns become document labels.

In [ ]:
df = pd.DataFrame(
    [[0, 1, 2, 3],
     [2, 0, 3, 1],
     [1, 2, 0, 3]],
    index=["BM25", "Semantic", "Cross-encoder"],
    columns=["Doc A", "Doc B", "Doc C", "Doc D"],
)

rf_from_df = RankFlow(df=df)
rf_from_df.plot()

## 3. Evaluating Retrieval Quality -- Metrics

When you know which documents are relevant (ground truth), rankflow computes standard retrieval metrics at each step:

| Metric | Description |
|--------|-------------|
| **Precision@K** | Fraction of top-K results that are relevant |
| **Recall@K** | Fraction of relevant docs found in top-K |
| **MRR** | Reciprocal rank of the first relevant document |
| **MAP** | Mean average precision across all relevant docs |
| **NDCG@K** | Normalized discounted cumulative gain |

Pass `relevant_chunks` to enable metrics, and `show_metrics=True` to overlay them on the plot.

In [ ]:
rf_metrics = RankFlow(
    ranks=ranks,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
    relevant_chunks=["Doc A", "Doc C"],
    show_metrics=True,
)
rf_metrics.plot()

print("Metrics per step (k=2):")
rf_metrics.metrics_df(k=2)

## 4. Graded Relevance

Binary relevance (relevant/not-relevant) is a simplification. In practice, human assessors assign graded judgments (e.g. 0 = irrelevant, 1 = marginally relevant, 3 = highly relevant). Pass `relevance_grades` to compute NDCG that accounts for these grades.

In [ ]:
rf_graded = RankFlow(
    ranks=ranks,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
    relevant_chunks=["Doc A", "Doc C"],
    relevance_grades={"Doc A": 3, "Doc C": 1},
)

print("Graded metrics (k=2):")
rf_graded.metrics_df(k=2)

## 5. Rank Change Analysis

`.summary_df()` shows per-document statistics: initial and final rank, total rank change, the biggest single-step gain and loss, and total displacement across all steps.

Set `show_deltas=True` to annotate the plot with rank changes between consecutive steps (green = improved, red = worsened).

In [ ]:
rf_deltas = RankFlow(
    ranks=ranks,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
    show_deltas=True,
)
rf_deltas.plot()

print("Per-document summary:")
rf_deltas.summary_df()

## 6. Top-K Filtering -- Focus on What Matters

When you have many candidate documents, the plot gets cluttered. Use `top_k` to display only the K documents that matter most.

The `top_k_mode` parameter controls which step is used to select the top-K:
- `"any"` (default) -- documents that appear in top-K at *any* step
- `"initial"` -- top-K at the first step only
- `"final"` -- top-K at the last step only

In [ ]:
ranks_large = np.array([
    [0, 1, 2, 3, 4, 5],
    [3, 0, 5, 1, 2, 4],
    [5, 2, 0, 1, 3, 4],
])
labels_large = ["Doc A", "Doc B", "Doc C", "Doc D", "Doc E", "Doc F"]

rf_topk = RankFlow(
    ranks=ranks_large,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=labels_large,
    top_k=3,
    top_k_mode="final",
)
rf_topk.plot()

## 7. A/B Pipeline Comparison

When evaluating different retrieval strategies, it helps to compare them side by side. `RankFlow.compare()` renders two pipelines on a shared y-axis so you can visually spot which one better promotes relevant documents.

In [ ]:
pipeline_a = RankFlow(
    ranks=np.array([[0, 1, 2, 3], [2, 0, 3, 1], [1, 2, 0, 3]]),
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
)
pipeline_b = RankFlow(
    ranks=np.array([[0, 1, 2, 3], [1, 2, 0, 3], [0, 3, 1, 2]]),
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
)

RankFlow.compare(pipeline_a, pipeline_b, labels=("Pipeline A", "Pipeline B"))

## 8. Multi-Query Evaluation with BatchRankFlow

A single query is rarely enough to judge pipeline quality. `BatchRankFlow` aggregates metrics across multiple queries and provides box plots and error-bar charts to show the distribution of retrieval quality.

Each `RankFlow` in the batch should have `relevant_chunks` set.

In [ ]:
queries = [
    RankFlow(
        ranks=np.array([[0, 1, 2, 3], [2, 0, 3, 1], [1, 2, 0, 3]]),
        step_labels=["BM25", "Semantic", "Cross-encoder"],
        chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
        relevant_chunks=["Doc A", "Doc C"],
    ),
    RankFlow(
        ranks=np.array([[0, 1, 2, 3], [1, 0, 3, 2], [0, 1, 2, 3]]),
        step_labels=["BM25", "Semantic", "Cross-encoder"],
        chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
        relevant_chunks=["Doc A", "Doc B"],
    ),
    RankFlow(
        ranks=np.array([[0, 1, 2, 3], [3, 2, 1, 0], [2, 3, 0, 1]]),
        step_labels=["BM25", "Semantic", "Cross-encoder"],
        chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
        relevant_chunks=["Doc C", "Doc D"],
    ),
]

batch = BatchRankFlow(queries)

In [ ]:
agg = batch.aggregate_metrics(k=2)

print("Overall mean metrics across all queries and steps:")
for key, val in agg["mean"].items():
    print(f"  {key}: {val:.4f}")

In [ ]:
batch.plot(k=2, metric="ndcg_at_k")

In [ ]:
batch.plot_metric_evolution(k=2)

## 9. Handling Dropped/Added Documents (NaN Ranks)

Some documents may not appear at every retrieval step -- for example, a re-ranker might filter out low-confidence candidates. Represent absent documents with `np.nan`; they will be rendered as dashed lines on the plot.

In [ ]:
ranks_with_nans = np.array([
    [0,    1,      2,    3],
    [1,    0,      np.nan, 2],
    [0,    np.nan, 1,    2],
])

rf_nan = RankFlow(
    ranks=ranks_with_nans,
    step_labels=["BM25", "Re-ranker", "Final"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
)
rf_nan.plot()

## 10. Customizing the Plot

All visual settings are controlled via `PlotConfig` fields. You can pass them as keyword arguments directly to the `RankFlow` constructor. Some useful options:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `title` | `"Rank evolution"` | Plot title |
| `fig_size` | auto | `(width, height)` in inches |
| `line_width` | `20` | Thickness of rank lines |
| `colors` | 16 default colors | List of colors for documents |
| `caption` | `"Re-ranking step"` | Text below the plot |

In [ ]:
rf_custom = RankFlow(
    ranks=ranks,
    step_labels=["BM25", "Semantic", "Cross-encoder"],
    chunk_labels=["Doc A", "Doc B", "Doc C", "Doc D"],
    title="My RAG Pipeline",
    fig_size=(10, 5),
    line_width=15,
    colors=["#e63946", "#457b9d", "#2a9d8f", "#e9c46a"],
    caption="Retrieval pipeline stages",
)
rf_custom.plot()

## 11. Exporting Results

Export rank data, metrics, and per-document summaries for downstream processing or reporting.

In [ ]:
print("As DataFrame:")
display(rf_metrics.to_dataframe())

print("\nAs dict (first 3 keys):")
d = rf_metrics.to_dict()
for key in list(d.keys())[:3]:
    print(f"  {key}: {d[key]}")

rf_metrics.to_json("rankflow_output.json")
print("\nSaved to rankflow_output.json")

## Interactive Plotly Backend

For interactive exploration, rankflow supports a Plotly backend with hover tooltips showing rank, score, and metric details:

```python
# pip install rankflow[interactive]
rf.iplot()  # or rf.plot(backend="plotly")
```